# Renovate Scope & Dependency Analysis
This notebook analyzes the `DEPENDENCY_SCOPE_REPORT.md` to audit the "Signal Purity" of the OS.
It parses the distribution of Core OS Runtime, Tooling, and Quarantine items.

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

# File path
report_path = 'DEPENDENCY_SCOPE_REPORT.md'

totals = {}
samples = []

try:
    with open(report_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Extract Totals
    totals_match = re.search(r'## Totals\n([\s\S]*?)\n##', content)
    if totals_match:
        for line in totals_match.group(1).strip().split('\n'):
            if line.strip():
                # Handle potential variation in format
                parts = line.strip().replace('- ', '').split(': ')
                if len(parts) == 2:
                    totals[parts[0]] = int(parts[1])

    # Extract Samples
    samples_match = re.search(r'## Top Evidence Samples\n([\s\S]*)', content)
    if samples_match:
        for line in samples_match.group(1).strip().split('\n'):
            if line.strip():
                # Format: - Name -> Category (local=X; total=Y; wiring=Z)
                match = re.match(r'- (.*?) -> (.*?) \(local=(\d+); total=(\d+); wiring=(.*?)\)', line.strip())
                if match:
                    samples.append({
                        'Component': match.group(1),
                        'Category': match.group(2),
                        'Local_Count': int(match.group(3)),
                        'Total_Count': int(match.group(4)),
                        'Wiring_Status': match.group(5)
                    })

    print(f"Loaded {len(samples)} sample records.")
    print("Totals from Report:", totals)

except FileNotFoundError:
    print(f"File {report_path} not found. Please ensure it exists in the workspace.")

In [ ]:
if totals:
    df_totals = pd.DataFrame(list(totals.items()), columns=['Category', 'Count'])
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Category', y='Count', data=df_totals, palette='viridis')
    plt.title('Dependency Scope Distribution (Core vs Quarantine)')
    plt.ylabel('Number of Dependencies')
    plt.xlabel('Scope Category')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Annotate bars
    for index, row in df_totals.iterrows():
        plt.text(index, row['Count'], str(row['Count']), color='black', ha="center", va="bottom")
        
    plt.show()
else:
    print("No total data loaded to visualize.")

In [ ]:
if samples:
    df_samples = pd.DataFrame(samples)
    print("Top 5 Evidence Samples:")
    # Using print to ensure output in basic runtimes, though display() is preferred in interactive NB
    print(df_samples.head(5).to_string(index=False))

    # Prepare data for plotting (Top 10 by Total Count)
    plot_df = df_samples.sort_values('Total_Count', ascending=False).head(10)

    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")
    
    # Bar chart of Usage
    chart = sns.barplot(x='Total_Count', y='Component', data=plot_df, hue='Category', dodge=False)
    
    plt.title('High-Usage Components: Scope Analyis')
    plt.xlabel('Total Usage Count (References)')
    plt.ylabel('Component Name')
    plt.tight_layout()
    plt.show()
else:
    print("No sample data available.")